In [ ]:
import os
import time
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv("cleaned_data.csv")
TARGET_REG = "SalePrice"

y_reg = df[TARGET_REG].astype(float)
y_clf = (y_reg > y_reg.median()).astype(int)
X = df.drop(columns=[TARGET_REG])

QUALITY_MAP = {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4}
ORDINAL_QUALITY_COLS = [
    "Exter_Qual", "Exter_Cond", "Bsmt_Qual", "Bsmt_Cond",
    "Heating_QC", "Kitchen_Qual", "Garage_Qual", "Garage_Cond",
]
for col in ORDINAL_QUALITY_COLS:
    X[col] = X[col].map(QUALITY_MAP)

ORDINAL_OTHER_MAPS = {
    "Lot_Shape":     {"Reg": 3, "IR1": 2, "IR2": 1, "IR3": 0},
    "Land_Slope":    {"Gtl": 2, "Mod": 1, "Sev": 0},
    "Bsmt_Exposure": {"Gd": 3, "Av": 2, "Mn": 1, "No": 0},
    "BsmtFin_Type_1":{"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0},
    "BsmtFin_Type_2":{"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0},
    "Garage_Finish": {"Fin": 2, "RFn": 1, "Unf": 0},
    "Paved_Drive":   {"Y": 2, "P": 1, "N": 0},
    "Utilities":     {"AllPub": 2, "NoSewr": 1, "NoSeWa": 0},
    "Functional":    {"Typ": 7, "Min1": 6, "Min2": 5, "Mod": 4, "Maj1": 3, "Maj2": 2, "Sev": 1, "Sal": 0},
    "Central_Air":   {"Y": 1, "N": 0},
}
for col, mapping in ORDINAL_OTHER_MAPS.items():
    X[col] = X[col].map(mapping)

nominal_cols = [c for c in X.columns if X[c].dtype == object or str(X[c].dtype).lower() == "str"]
X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)
X = X.fillna(X.median(numeric_only=True))
X = X.astype(float)

X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Loaded and re-encoded data. X_train_scaled: {X_train_scaled.shape}, "
      f"X_test_scaled: {X_test_scaled.shape}")

feature_names = X.columns.tolist()

# Baseline Logistic Regression from Part 2 (needed for the CV comparison in Task 5)
log_reg_baseline = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced",
                                       random_state=RANDOM_STATE)
log_reg_baseline.fit(X_train_scaled, y_clf_train)

### TASK 1 — Decision Tree baseline (unconstrained)


In [ ]:
dt_unconstrained = DecisionTreeClassifier(max_depth=None, random_state=RANDOM_STATE)
dt_unconstrained.fit(X_train_scaled, y_clf_train)

train_acc_unconstrained = accuracy_score(y_clf_train, dt_unconstrained.predict(X_train_scaled))
test_acc_unconstrained = accuracy_score(y_clf_test, dt_unconstrained.predict(X_test_scaled))
print(f"Train accuracy: {train_acc_unconstrained:.4f}")
print(f"Test accuracy:  {test_acc_unconstrained:.4f}")
print(f"Train-test gap: {train_acc_unconstrained - test_acc_unconstrained:.4f}")


### TASK 2 — Controlled Decision Tree

In [ ]:
dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=RANDOM_STATE)
dt_controlled.fit(X_train_scaled, y_clf_train)

train_acc_controlled = accuracy_score(y_clf_train, dt_controlled.predict(X_train_scaled))
test_acc_controlled = accuracy_score(y_clf_test, dt_controlled.predict(X_test_scaled))
print(f"Train accuracy: {train_acc_controlled:.4f}")
print(f"Test accuracy:  {test_acc_controlled:.4f}")
print(f"Train-test gap: {train_acc_controlled - test_acc_controlled:.4f}")
print(f"\nGap comparison -> unconstrained: {train_acc_unconstrained - test_acc_unconstrained:.4f}, "
      f"controlled: {train_acc_controlled - test_acc_controlled:.4f}")

### TASK 3 — Gini vs Entropy comparison

In [ ]:
dt_gini = DecisionTreeClassifier(max_depth=5, criterion="gini", random_state=RANDOM_STATE)
dt_gini.fit(X_train_scaled, y_clf_train)
test_acc_gini = accuracy_score(y_clf_test, dt_gini.predict(X_test_scaled))

dt_entropy = DecisionTreeClassifier(max_depth=5, criterion="entropy", random_state=RANDOM_STATE)
dt_entropy.fit(X_train_scaled, y_clf_train)
test_acc_entropy = accuracy_score(y_clf_test, dt_entropy.predict(X_test_scaled))

print(f"Gini test accuracy:    {test_acc_gini:.4f}")
print(f"Entropy test accuracy: {test_acc_entropy:.4f}")

### TASK 4 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
rf.fit(X_train_scaled, y_clf_train)

rf_train_acc = accuracy_score(y_clf_train, rf.predict(X_train_scaled))
rf_test_acc = accuracy_score(y_clf_test, rf.predict(X_test_scaled))
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]
rf_auc = roc_auc_score(y_clf_test, rf_proba)

print(f"Train accuracy: {rf_train_acc:.4f}")
print(f"Test accuracy:  {rf_test_acc:.4f}")
print(f"Test ROC-AUC:   {rf_auc:.4f}")

importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
top5_features = importances.head(5)
print("\nTop 5 features by importance:")
print(top5_features)

### TASK 4a — Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
                                 random_state=RANDOM_STATE)
gb.fit(X_train_scaled, y_clf_train)

gb_train_acc = accuracy_score(y_clf_train, gb.predict(X_train_scaled))
gb_test_acc = accuracy_score(y_clf_test, gb.predict(X_test_scaled))
gb_proba = gb.predict_proba(X_test_scaled)[:, 1]
gb_auc = roc_auc_score(y_clf_test, gb_proba)

print(f"Train accuracy: {gb_train_acc:.4f}")
print(f"Test accuracy:  {gb_test_acc:.4f}")
print(f"Test ROC-AUC:   {gb_auc:.4f}")

### TASK 4b — Feature ablation study

In [ ]:
lowest5_features = importances.tail(5)
print("5 lowest-importance features (to be removed):")
print(lowest5_features)

drop_cols = lowest5_features.index.tolist()
keep_idx = [i for i, f in enumerate(feature_names) if f not in drop_cols]

X_train_reduced = X_train_scaled[:, keep_idx]
X_test_reduced = X_test_scaled[:, keep_idx]

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
rf_reduced.fit(X_train_reduced, y_clf_train)
rf_reduced_proba = rf_reduced.predict_proba(X_test_reduced)[:, 1]
rf_reduced_auc = roc_auc_score(y_clf_test, rf_reduced_proba)

print(f"\nFull-model test AUC (all {len(feature_names)} features):    {rf_auc:.4f}")
print(f"Reduced-model test AUC ({len(feature_names)-5} features): {rf_reduced_auc:.4f}")
print(f"AUC change from removing 5 lowest-importance features: {rf_reduced_auc - rf_auc:+.4f}")

### TASK 5 — Cross-validated comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models_for_cv = {
    "Logistic Regression": log_reg_baseline,
    "Decision Tree (max_depth=5)": dt_controlled,
    "Random Forest": rf,
    "Gradient Boosting": gb,
}

cv_results = []
for name, model in models_for_cv.items():
    scores = cross_val_score(model, X_train_scaled, y_clf_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_results.append({"Model": name, "CV_Mean_AUC": scores.mean(), "CV_Std_AUC": scores.std()})
    print(f"{name}: mean AUC = {scores.mean():.4f}, std = {scores.std():.4f}")

cv_results_df = pd.DataFrame(cv_results)
print("\nCross-validation summary table:")
print(cv_results_df.to_string(index=False))

### TASK 6 — Hyperparameter tuning with GridSearchCV

In [ ]:

param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5],
}

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=RANDOM_STATE),
)

grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1
)

# Fit on UNSCALED X_train -- the pipeline itself handles imputation + scaling
t0 = time.time()
grid_search.fit(X_train, y_clf_train)
elapsed = time.time() - t0

n_configs = 1
for v in param_grid.values():
    n_configs *= len(v)
n_fits = n_configs * cv.get_n_splits()

print(f"Grid search completed in {elapsed:.1f}s")
print(f"Total parameter combinations: {n_configs}")
print(f"Total fits (combinations x folds): {n_fits}")
print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV score (roc_auc): {grid_search.best_score_:.4f}")

best_pipeline = grid_search.best_estimator_
best_pipeline_test_auc = roc_auc_score(
    y_clf_test, best_pipeline.predict_proba(X_test)[:, 1]
)
print(f"Best pipeline test-set AUC: {best_pipeline_test_auc:.4f}")

### TASK 6 (extension) — Manual learning curve using the tuned best_pipeline

In [ ]:
from sklearn.base import clone

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
learning_curve_rows = []

for f in fractions:
    n_rows = int(f * len(X_train))
    X_sub = X_train.iloc[:n_rows]
    y_sub = y_clf_train.iloc[:n_rows]

    pipeline_f = clone(best_pipeline)
    pipeline_f.fit(X_sub, y_sub)

    train_auc = roc_auc_score(y_sub, pipeline_f.predict_proba(X_sub)[:, 1])
    test_auc = roc_auc_score(y_clf_test, pipeline_f.predict_proba(X_test)[:, 1])

    learning_curve_rows.append({
        "Training_fraction": f,
        "n_rows": n_rows,
        "Training_AUC": train_auc,
        "Test_AUC": test_auc,
    })
    print(f"fraction={f:.1f} (n={n_rows}): Training AUC={train_auc:.4f}, Test AUC={test_auc:.4f}")

learning_curve_df = pd.DataFrame(learning_curve_rows)
print("\nLearning curve table:")
print(learning_curve_df.to_string(index=False))

### TASK 7 — Serialize the best model

In [ ]:
joblib.dump(best_pipeline, "best_model.pkl")
print("Saved best_pipeline -> best_model.pkl")

# --- Reload-and-predict demonstration block (required, >= 5 lines) ---
loaded_model = joblib.load("best_model.pkl")                  # 1
test_row_1 = X_test.iloc[[0]]                                 # 2  hand-picked real test row
test_row_2 = X_test.iloc[[1]]                                 # 3  hand-picked real test row
pred_1 = loaded_model.predict(test_row_1)                     # 4
pred_2 = loaded_model.predict(test_row_2)                     # 5
print(f"Reloaded model prediction on test row 1: {pred_1[0]} "
      f"(actual label: {y_clf_test.iloc[0]})")
print(f"Reloaded model prediction on test row 2: {pred_2[0]} "
      f"(actual label: {y_clf_test.iloc[1]})")
print("Reload-and-predict block ran without errors.")

In [ ]:
summary_table = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree (max_depth=5)",
              "Random Forest", "Gradient Boosting", "Tuned RF (GridSearchCV)"],
    "CV_Mean_AUC": [
        cv_results_df.loc[cv_results_df.Model == "Logistic Regression", "CV_Mean_AUC"].values[0],
        cv_results_df.loc[cv_results_df.Model == "Decision Tree (max_depth=5)", "CV_Mean_AUC"].values[0],
        cv_results_df.loc[cv_results_df.Model == "Random Forest", "CV_Mean_AUC"].values[0],
        cv_results_df.loc[cv_results_df.Model == "Gradient Boosting", "CV_Mean_AUC"].values[0],
        grid_search.best_score_,
    ],
    "CV_Std_AUC": [
        cv_results_df.loc[cv_results_df.Model == "Logistic Regression", "CV_Std_AUC"].values[0],
        cv_results_df.loc[cv_results_df.Model == "Decision Tree (max_depth=5)", "CV_Std_AUC"].values[0],
        cv_results_df.loc[cv_results_df.Model == "Random Forest", "CV_Std_AUC"].values[0],
        cv_results_df.loc[cv_results_df.Model == "Gradient Boosting", "CV_Std_AUC"].values[0],
        np.nan,  # GridSearchCV doesn't directly expose std of the best combo's folds here
    ],
    "Test_AUC": [
        roc_auc_score(y_clf_test, log_reg_baseline.predict_proba(X_test_scaled)[:, 1]),
        roc_auc_score(y_clf_test, dt_controlled.predict_proba(X_test_scaled)[:, 1]),
        rf_auc,
        gb_auc,
        best_pipeline_test_auc,
    ],
})
print("\n=== FINAL SUMMARY TABLE (copy into README) ===")
print(summary_table.to_string(index=False))

summary_table.to_csv(os.path.join(OUT_DIR, "model_comparison_summary.csv"), index=False)
cv_results_df.to_csv(os.path.join(OUT_DIR, "cv_results.csv"), index=False)
learning_curve_df.to_csv(os.path.join(OUT_DIR, "learning_curve.csv"), index=False)
top5_features.to_csv(os.path.join(OUT_DIR, "top5_feature_importance.csv"))
pd.DataFrame({
    "full_model_auc": [rf_auc],
    "reduced_model_auc": [rf_reduced_auc],
    "auc_change": [rf_reduced_auc - rf_auc],
}).to_csv(os.path.join(OUT_DIR, "ablation_study.csv"), index=False)

print(f"\nAll summary tables saved to ./{OUT_DIR}/")
print("best_model.pkl saved in current directory.")
print("Done.")